In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

In [3]:
from model_implementations.LSTMLinearRegression import LSTM as LSTMLinearRegression
from model_implementations.LSTMFeedForwardNeuralNetwork import LSTM as LSTMFeedForwardNeuralNetwork
from model_implementations.LSTMSelfAttention import LSTM_with_SA as LSTMSelfAttention

In [4]:
pretrained_models_dir = "./pth_models/pretrained_on_11exp_with_shift"
onnx_models_dir = "../onnx_models"

# Saving LSTM+LR as ONNX model

In [5]:
# Optimal hyperparameters
seq_len_lr = 4
hidden_size = 3
learning_rate = 0.04024962074596677
factor = 0.8016939043270802
batch_size = 3

lstm_lr_state_dict = torch.load(f"{pretrained_models_dir}/LSTM+LR_10trial_169ep_MAE-0.99_0.96_0.88.pth", map_location=torch.device('cpu'))['model_state_dict']
lstm_lr_model = LSTMLinearRegression(input_size=2, hidden_size=hidden_size, device=torch.device('cpu'))
lstm_lr_model.load_state_dict(lstm_lr_state_dict)
lstm_lr_model.eval()

LSTM(
  (lstm): LSTM(2, 3, batch_first=True)
  (relu): ReLU()
  (fc): Linear(in_features=3, out_features=1, bias=True)
)

In [ ]:
fake_tensor = torch.randn((1, seq_len_lr, 2), dtype=torch.float32)

# torch.onnx.export(
#     lstm_lr_model,
#     (fake_tensor,),
#     f"{onnx_models_dir}/lstm_lr.onnx",
#     input_names=["input"],
#     output_names=["output"],
# )

# Saving LSTM+FFNN as ONNX model

In [5]:
# Optimal hyperparameters
seq_len_ffnn = 8
hidden_size = 6
learning_rate = 0.06418282214056456
scheduler_factor = 0.7675368352048451
batch_size = 3
activation_func = torch.nn.LogSigmoid()
hidden_layer_size = 2

lstm_ffnn_state_dict = torch.load(
    f"{pretrained_models_dir}/LSTM_fine_tuning_16trial_79ep_MAE-0.636_0.643_0.639.pth",
    map_location=torch.device('cpu')
    )['model_state_dict']

filtered_state_dict = {k: v for k, v in lstm_ffnn_state_dict.items() if not k.startswith('offsets')}
print(f"✅ Загружено {len(filtered_state_dict)} весов из {len(lstm_ffnn_state_dict)} (offsets исключены)")

lstm_ffnn_model = LSTMFeedForwardNeuralNetwork(
    input_size=2,
    hidden_size=hidden_size,
    activation_func=activation_func,
    hidden_layer_size=hidden_layer_size,
    device=torch.device('cpu')
    )

# 🔹 Загружаем с strict=False (на случай других несовпадений)
missing_keys, unexpected_keys = lstm_ffnn_model.load_state_dict(filtered_state_dict, strict=False)

if missing_keys:
    print(f"⚠️  Отсутствуют ключи: {missing_keys}")
if unexpected_keys:
    print(f"⚠️  Неожиданные ключи: {unexpected_keys}")

lstm_ffnn_model.eval()

✅ Загружено 10 весов из 11 (offsets исключены)


LSTM(
  (activation_func): LogSigmoid()
  (lstm): LSTM(2, 6, batch_first=True)
  (fc1): Linear(in_features=6, out_features=2, bias=True)
  (fc2): Linear(in_features=2, out_features=2, bias=True)
  (fc3): Linear(in_features=2, out_features=1, bias=True)
)

In [8]:
fake_tensor = torch.randn((1, seq_len_ffnn, 2), dtype=torch.float32)

torch.onnx.export(
    lstm_ffnn_model,
    (fake_tensor,),
    f"{onnx_models_dir}/lstm_ffnn_shift_version.onnx",
    input_names=["input"],
    output_names=["output"],
)

# Saving LSTM+SA as ONNX model

In [10]:
# Optimal hyperparameters
hidden_size = 7
seq_len_sa = 7
self_attention_dim = 3
activation_func = torch.nn.Sigmoid()
hidden_layer_size = 3
learning_rate = 0.16300957594151824
weight_decay = 0.0029507235557957225
batch_size = 3
warmup_steps = 6
min_lr = 1e-06

lstm_sa_state_dict = torch.load(f"{pretrained_models_dir}/LSTM+SA_59trial_176ep_MAE-1.34_0.92_0.96.pth", map_location=torch.device('cpu'))['model_state_dict']
lstm_sa_model = LSTMSelfAttention(input_size=2, seq_len=seq_len_sa, hidden_size=hidden_size, model_sa_dim=self_attention_dim, activation_func=activation_func, hidden_layer_size=hidden_layer_size, device=torch.device('cpu'))
lstm_sa_model.load_state_dict(lstm_sa_state_dict)
lstm_sa_model.eval()

LSTM_with_SA(
  (activation_func): Sigmoid()
  (lstm): LSTM_Block(
    (lstm): LSTM(2, 7, batch_first=True)
  )
  (sa): SelfAttention(
    (query): Linear(in_features=7, out_features=3, bias=True)
    (key): Linear(in_features=7, out_features=3, bias=True)
    (value): Linear(in_features=7, out_features=3, bias=True)
    (softmax): Softmax(dim=2)
  )
  (fc1): Linear(in_features=21, out_features=3, bias=True)
  (fc2): Linear(in_features=3, out_features=1, bias=True)
)

In [ ]:
fake_tensor = torch.randn((1, seq_len_sa, 2), dtype=torch.float32)

# torch.onnx.export(
#     lstm_sa_model,
#     (fake_tensor,),
#     f"{onnx_models_dir}/lstm_sa.onnx",
#     input_names=["input"],
#     output_names=["output"],
# )

# ONNX inference

In [12]:
import onnxruntime as ort

In [18]:
model_name = "lstm_sa.onnx"
seq_len = seq_len_sa
model = lstm_sa_model

session = ort.InferenceSession(f"{onnx_models_dir}/{model_name}")

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

display(input_name)
display(output_name)

input_tensor = np.random.rand(1, seq_len, 2).astype(np.float32)

onns_outputs = session.run([output_name], {input_name: input_tensor})
onnx_model_output = onns_outputs[0].flatten()

with torch.no_grad():
    original_outputs = model(torch.from_numpy(input_tensor))
    original_model_output = original_outputs.flatten()

print(f"{onnx_model_output = }")
print(f"{original_model_output = }")

assert torch.isclose(torch.from_numpy(onnx_model_output), original_model_output)

'input'

'output'

onnx_model_output = array([1.8216002], dtype=float32)
original_model_output = tensor([1.8216])
